<a href="https://colab.research.google.com/github/evandethier/arctic_rivers/blob/master/code/8_thaw_slump_batch_image_export.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##**Introduction**

### **i. Initialize/authenticate project**

In [ ]:
# Import necessary libraries
import ee, geemap

# Connect to GEE
ee.Authenticate()
ee.Initialize(project='remote2023-412115')

# Import the `drive` library
from google.colab import drive

# Then we can "mount" the drive.
# This means we can work with files stored there.
drive.mount('/content/drive')

In [ ]:
# Import the libraries for data analysis
import pandas as pd
import numpy as np
!pip install lets_plot
from lets_plot import *
from datetime import datetime
import time
import os

In [ ]:
# Date range
startDate = '2005-01-01'
endDate = '2025-12-31'
# Day of year restrictions
start_yday = 180
end_yday = 270
# Determine number of images to export
num_export_imgs = 'all'

# If the export should just be RGB
# img_type = 'vis'
# If export should be all bands:
img_type = 'all'
# num_export_imgs = 'pair'

# Export images
yes_no_image_export = True
# yes_no_image_export = False
# Export table
yes_no_table_export = False

In [ ]:
import re

# Define area of interest name
location_name = 'taymyr_peninsula'
wd_root = '/content/drive/MyDrive/Colab Notebooks/arctic_erosion_gee'
wd_imports = f'{wd_root}/arctic_erosion_imports/'
wd_imports_imgs = f'{wd_imports}taymyr_image_exports/'

# FeatureCollection
aoi_polys = ee.FeatureCollection('projects/ee-edethier/assets/arctic_erosion/taymyr_slumps_2024_20250620_120px_60m_all')

site_id_field = 'label'
# Unique IDs for aois
aoi_names = aoi_polys.aggregate_array(site_id_field).distinct().getInfo()
n_features = len(aoi_names)
print(n_features)

# display(aoi_names.getInfo()[1:20])

# Make aoi into an image
aoi_label_mask = ee.Image(0).byte().paint(aoi_polys,1).selfMask().rename('label2024')

In [ ]:
year_dummy = 2018
display(ee.String('label').cat(ee.String(ee.Number(year_dummy))))


In [ ]:
slumps_2024 = ee.FeatureCollection('projects/ee-edethier/assets/arctic_erosion/taymyr_slumps_2024_20250620_120px_60m_all')
slumps_2023 = ee.FeatureCollection('projects/ee-edethier/assets/arctic_erosion/taymyr_slumps_2023_20250620')
slumps_2022 = ee.FeatureCollection('projects/ee-edethier/assets/arctic_erosion/taymyr_slumps_2022_20250620')
slumps_2021 = ee.FeatureCollection('projects/ee-edethier/assets/arctic_erosion/taymyr_slumps_2021_20250620')
slumps_2020 = ee.FeatureCollection('projects/ee-edethier/assets/arctic_erosion/taymyr_slumps_2020_20250620')
slumps_2019 = ee.FeatureCollection('projects/ee-edethier/assets/arctic_erosion/taymyr_slumps_2019_20250620')
slumps_2018 = ee.FeatureCollection('projects/ee-edethier/assets/arctic_erosion/taymyr_slumps_2018_20250620')
slumps_2017 = ee.FeatureCollection('projects/ee-edethier/assets/arctic_erosion/taymyr_slumps_2017_20250620')
slumps_2016 = ee.FeatureCollection('projects/ee-edethier/assets/arctic_erosion/taymyr_slumps_2016_20250620')

def getArea(feature):
  return feature.set('area_km2', ee.Number(feature.geometry().area()).divide(1e6))

all_slumps_fc = (ee.FeatureCollection([
  slumps_2018.map(lambda x: x.set('year', 2018)),
  slumps_2019.map(lambda x: x.set('year', 2019)),
  slumps_2020.map(lambda x: x.set('year', 2020)),
  slumps_2021.map(lambda x: x.set('year', 2021)),
  slumps_2022.map(lambda x: x.set('year', 2022)),
  slumps_2023.map(lambda x: x.set('year', 2023)),
  slumps_2024.map(lambda x: x.set('year', 2024))
  ])
).flatten().map(getArea)


def getPaint(year):
  fc = all_slumps_fc.filter(ee.Filter.eq('year', year))
  # year_sel = ee.FeatureCollection(fc).first()
  labeled_image = ee.Image(0).byte().paint(fc,1).selfMask()
  return labeled_image.set('year', year)


all_slumps_painted = ee.ImageCollection(ee.List.sequence(2018, 2024, 1).map(getPaint))

display(all_slumps_painted)

In [ ]:
# OPTIONAL -- VISUALIZE FEATURES ON A MAP
Map = geemap.Map()

Map.add_basemap("Esri.WorldImagery")
Map.add_ee_layer(aoi_polys, {'color': 'red'}, 'Export locations')
Map.add_ee_layer(aoi_label_mask, {'palette': 'blue'}, 'AOI mask')

# Center AOI and zoom
Map.centerObject(aoi_polys, 11)

Map.add_ee_layer(ee.Image(all_slumps_painted.filter(ee.Filter.eq('year', 2023)).first()), {'palette': 'yellow'}, 'slumps')
display(Map)

###**Define thresholds**

In [ ]:
# Define a variable called cloud_thresh to store your cloud threshold
cloud_thresh = 30

# Do the same for NDVI (vegetation)
ndvi_thresh = 0.5

# For NDWI (water)
ndwi_thresh = 0

# And for NDSI (snow)
ndsi_thresh = 0.42

# What %clouds can an image be to export it
cloud_ratio_thresh = 0.15


###**Functions for processing Landsat data**

In [ ]:
# Functions for import, depending on satellite (5 & 7 vs. 8 & 9)
def import_LS57_data(image):
        non_temp_bands = image.select(['SR_B.*']).multiply(0.0000275).add(-0.2).multiply(10000)
        temp_band = image.select('ST_B6').multiply(0.00341802).add(149)
        qa_band = image.select('QA_PIXEL')
        new_image = (image.select('SR_CLOUD_QA')
                              .addBands(non_temp_bands)
                              .addBands(temp_band)
                              .addBands(qa_band.rename('pixel_qa'))
                              .copyProperties(image)
        )

        final_image = ee.Image(new_image).select(["SR_B.*", "ST_B.*",'pixel_qa'], ['B1','B2','B3','B4','B5','B7','temp_K', 'pixel_qa'])
        return final_image

#  For Landsat 5-9
def import_LS89_data(image):
        non_temp_bands = image.select(['SR_B.*']).multiply(0.0000275).add(-0.2).multiply(10000)
        temp_band = image.select('ST_B10').multiply(0.00341802).add(149)
        qa_band = image.select('QA_PIXEL')
        new_image = (image.select('SR_QA_AEROSOL')
                              .addBands(non_temp_bands
                                        .select(['SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7'])
                                        )
                              .addBands(temp_band)
                              .addBands(qa_band.rename('pixel_qa'))
                              .copyProperties(image)
        )

        # return(ee.Image(new_image).select(["SR_B.*", "ST_B.*", 'pixel_qa'], ['B0','B1','B2','B3','B4','B5','B7','B6', 'pixel_qa']))
        return(ee.Image(new_image).select(["SR_B.*", "ST_B.*", 'pixel_qa'], ['B1','B2','B3','B4','B5','B7','temp_K', 'pixel_qa']))

# Add formatted date, month, and year to each image
def getDate(image):
  date = image.date()
  month = date.get('month')
  year = date.get('year')
  formatted_date = date.format('YYYY-MM-dd')
  return image.set('date', formatted_date, 'month', month, 'year', year)

###**Functions to get clouds, water, vegetation, and snow**

In [ ]:
# Apply removeClouds > getWater > getNDWI in order
def getClouds(image):
  qaBand = image.select('pixel_qa')
  # Create bitmasks
  cloudConfidenceMask = 3 << 8
  cloudShadowConfidenceMask = 3 << 9

  # Apply the bitmask and check conditions
  cloudMask = qaBand.bitwiseAnd(cloudConfidenceMask).gt(256)
  shadowMask = qaBand.bitwiseAnd(cloudShadowConfidenceMask).gt(1024)

  # Combine the masks
  clouds = cloudMask.Or(shadowMask).rename('clouds').selfMask()

  return clouds



def getWater(image):
  # Get clouds for the image using getClouds function
  clouds = getClouds(image)

  # calculate NDWI
  ndwi = image.normalizedDifference(['B2','B5'])
  # Use NDWI to ID water and B4 to remove snow/ice and glare
  water = (ee.Image(0)
        .updateMask(ndwi.gt(0.1))
        .updateMask(image.select('B4').lt(2000))
        .rename('water')
        .addBands(clouds))

  return water.copyProperties(image)


def getNDVI(image):
  # Get clouds for the image using getClouds function
  clouds = getClouds(image)

  # Calculate NDVI
  ndvi = image.normalizedDifference(['B4','B3']).rename('vegetation')

  # Mask NDVI using threshold
  ndvi = (ndvi.updateMask(ndvi.gt(ndvi_thresh))
              .addBands(clouds)
  )

  return ndvi.copyProperties(image)

def getNDSI(image):
  # Get clouds for the image using getClouds function
  clouds = getClouds(image)

  b3_thresh = image.select('B3').gt(2800)
  ndsi = image.normalizedDifference(['B2','B5'])

  ndsi = (ndsi.gt(ndsi_thresh)
            .selfMask().rename('snow')
            .updateMask(b3_thresh)
            )

  ndsi = ndsi.addBands(clouds)

  return ndsi.copyProperties(image)

def addClouds_landsat(image):
  qaBand = image.select('pixel_qa')
  # Create bitmasks
  cloudConfidenceMask = 3 << 8
  cloudShadowConfidenceMask = 3 << 9

  # Apply the bitmask and check conditions
  cloudMask = qaBand.bitwiseAnd(cloudConfidenceMask).gt(256)
  shadowMask = qaBand.bitwiseAnd(cloudShadowConfidenceMask).gt(1024)

  # Combine the masks
  clouds = cloudMask.Or(shadowMask).rename('clouds')

  return image.addBands(clouds)

###**Feature analysis functions**

In [ ]:
# Export images from label year
# Function to get clear and cloudy pixels from a feature collection
# (Must have sampled an image first)
def clearEnough(feature):
  baseline = ee.Number(feature.get('baseline')).multiply(cloud_ratio_thresh)
  clouds = ee.Number(feature.get('clouds'))
  snow = ee.Number(feature.get('snow'))
  clear_enough = baseline.gt(clouds).multiply(baseline.gt(snow))
  return feature.set('clear_enough', clear_enough)

###Add visualization parameters

In [ ]:
# Choose bands for visualization
bands = ['B3','B2','B1']

# Make a variable storing the visualization parameters
# (Data type: Dictionary)
## YOUR CODE GOES HERE
im_vis = {"bands": bands, "max": 2500, "min": 0}

##**Import Data**

###**Import image collection, filter for clouds, and apply processing function**

In [ ]:
# Map the processing function over the image collection:
# Gets Landsat 5, 8, and 9 all into the same data type.

# Landsat 5, 7, 8, and 9
ls5 = (ee.ImageCollection("LANDSAT/LT05/C02/T1_L2")
            .filterDate(startDate, endDate)
            .filter(ee.Filter.lt('CLOUD_COVER', cloud_thresh))
            .filter(ee.Filter.calendarRange(start_yday, end_yday, 'day_of_year'))
            ).map(import_LS57_data)

ls7 = (ee.ImageCollection("LANDSAT/LE07/C02/T1_L2")
            .filterDate('1999-01-01', '2003-05-30')
            .filterDate(startDate, endDate)
            .filter(ee.Filter.lt('CLOUD_COVER', cloud_thresh))
            .filter(ee.Filter.calendarRange(start_yday, end_yday, 'day_of_year'))
            ).map(import_LS57_data)

ls8 = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
            .filterDate(startDate, endDate)
            .filter(ee.Filter.lt('CLOUD_COVER', cloud_thresh))
            .filter(ee.Filter.calendarRange(start_yday, end_yday, 'day_of_year'))
            ).map(import_LS89_data)

ls9 = (ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
            .filterDate(startDate, endDate)
            .filter(ee.Filter.lt('CLOUD_COVER', cloud_thresh))
            .filter(ee.Filter.calendarRange(start_yday, end_yday, 'day_of_year'))
            ).map(import_LS89_data)

# Merge collections
im_coll_all = ls5.merge(ls8).merge(ls7).merge(ls9).map(getDate).map(addClouds_landsat)



###**Sentinel-2 data**

In [ ]:
# First, import data to improve image quality
# Cloud masking dataset
# https://medium.com/google-earth/more-accurate-and-flexible-cloud-masking-for-sentinel-2-images-766897a9ba5f
cloud_prob_thresh = 50
s2_mission = "COPERNICUS/S2_HARMONIZED"
def addClouds_s2(image):
  image_id = image.get('system:index')
  clouds_ic = (ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')
                        .filterBounds(image.geometry().centroid())
                        .filterDate(image.date().advance(-1,'day'),image.date().advance(1,'day'))
                        .filter(ee.Filter.eq('system:index', image_id))
                        )
  clouds = clouds_ic.first().gt(cloud_prob_thresh)
  # not_clouds = (clouds.lte(cloud_prob_thresh).rename('not_clouds')
  #                         .reproject({'crs': image.select(0).projection(), 'scale': 2000})
  # )

  # cloud_added = image.addBands(clouds.rename('clouds'))
  # n_bands = cloud_added.bandNames().size()
  return image.addBands(clouds.rename('clouds'))
  # .set('n_bands', n_bands)


s2_ic = (ee.ImageCollection(s2_mission)
          .filterDate('2018-06-01', '2025-07-01')
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_thresh))
            .filter(ee.Filter.calendarRange(start_yday, end_yday, 'day_of_year'))
          .select(['B2','B3','B4','B5','B8','B8A','B11','B12'])
          .map(addClouds_s2)
          .map(getDate)
)

In [ ]:
# # display(s2_ic.filterBounds(aoi_polys.first().geometry()).sort('system:time_start', False).first().getInfo())
# display(ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY')
#               .filterBounds(s2_ic.first().geometry())
#               .filterDate(s2_ic.first().date().advance(-1,'day'), s2_ic.first().date().advance(1,'day'))
#               .filter(ee.Filter.eq('system:index',s2_ic.first().get('system:index')))
#               .first()
#               )

# display(s2_ic.filterBounds(aoi_polys).first())
# # display(ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY').filter(ee.Filter.eq('system:index', '20250714T060639_20250714T060634_T47XND')).size().eq(0))

In [ ]:
def list_my_ops():
    """
    Return a list of all operations (tasks) owned by the current credentials.
    `limit` is the maximum number of results the server will return in one call.
    """
    # Low-level API call; returns a list of LRO-style dictionaries.
    return ee.data.listOperations()

PENDING_STATES = {"PENDING", "QUEUED", "READY", "RUNNING", "ACTIVE"}

def pending_count():
    ops = list_my_ops()
    return sum(
        1
        for op in ops
        if (not op.get("done", False))               # still in progress
        and op.get("metadata", {}).get("state") in PENDING_STATES
    )

pending_count()

In [ ]:
def export_image_batch(export_images, export_dates, site_name,
                       region,export_folder = 'GEE_test_exports',
                       export_folder_path='/content/drive/MyDrive/Colab Notebooks/',
                       n_export_images = 500,
                       crs='EPSG:3857', scale = 30,
                       vis_params = {"bands": ['B4','B3','B2'], "min": 0, "max": 2000},
                       safety_pad = 300, poll_sec = 300, start_tasks=False):

      # Create metadata file
      metadata_file = pd.DataFrame({
          'date': pd.to_datetime(export_dates),
          'site_no': site_name,
          'baseline': export_images.aggregate_array('baseline').getInfo(),
          'clouds': export_images.aggregate_array('clouds').getInfo(),
          'clouds_img': export_images.aggregate_array('CLOUDY_PIXEL_PERCENTAGE').getInfo(),
          'id': export_images.aggregate_array('system:index').getInfo()
      })

      metadata_file = metadata_file.assign(
          yday = metadata_file['date'].dt.day_of_year,
          year = metadata_file['date'].dt.year,
          yday7 = metadata_file['date'].dt.day_of_year - metadata_file['date'].dt.day_of_year%7
      )

      # display(metadata_file)

      # Sort by CLOUDY_PIXEL_PERCENTAGE
      metadata_file = metadata_file.sort_values(by = ['year','clouds_img'])
      # Rank by clouds at AOI for each period
      metadata_file['best_image'] = (metadata_file
                                    .groupby(['year','yday7'])['clouds']
                                    .rank(method='first', ascending=True)
      )

      # display(metadata_file.groupby(['year','yday20'])['clouds'].rank(method='first', ascending=True))
      # Choose image from each period to export
      metadata_file_best = metadata_file[metadata_file['best_image'] == 1].copy().sort_values(by=['date']).reset_index()

      while True:
        n_tasks = pending_count()             # READY + RUNNING
        free_slots = 3000 - n_tasks - safety_pad

        if free_slots >= n_export_images:
            print(f"✔ Enough room ({free_slots} slots) to queue {n_export_images} images.")
            break
        else:
            print(f"⏳ Only {free_slots} free slots; need {n_export_images}. Sleeping {poll_sec}s.")
            time.sleep(poll_sec)

      if start_tasks:
        folder=f'{export_folder_path}/s{site_name}'
        os.makedirs(folder, exist_ok=True)
        metadata_file.to_csv(f'{folder}/{satellite_abrv}_{site_name}_metadata.csv',index=False)

      # display(export_images)
      # Loop over the list and export each image.
      for _id, _date in zip(metadata_file_best['id'], metadata_file_best['date']):
        export_image = ee.Image(export_images.filter(ee.Filter.eq('system:index', _id)).first())

        # Define the export task.
        if img_type == 'vis':
          export_image = export_image.visualize(**vis_params)
        image_export = ee.batch.Export.image.toDrive(
            image=export_image,
            description=f'{satellite_abrv}_{site_name}_{_id}_{_date}',
            fileNamePrefix=f'{satellite_abrv}_{site_name}_{_id}_{_date}',
            folder=f's{site_name}',
            scale=scale_sel,
            region=export_buffer,
            crs='EPSG:3857',
            fileFormat='GeoTIFF'
        )

        if start_tasks:
          # Start the image export task.
          image_export.start()
          # print(f'Exporting Landsat Image {name_sel} to Google Drive.')
          # print(f'export image {j}')

      print(f'Exporting images and metadata for {name_sel} to Google Drive.')
      # return metadata_file



In [ ]:
# satellite = 'Landsat'
satellite = 'Sentinel-2'

scale_sel = 30 # default scale

if satellite == 'Landsat':
  im_coll_all = im_coll_all
  # Specify scale
  scale_sel = 30
  cast_dict = {
                'B1':'int16',
                'B2':'int16',
                'B3':'int16',
                'B4':'int16',
                'B5':'int16',
                'B7':'int16',
                'temp_K':'int16',
                'clouds':'int16',
                'label2024':'int16',
                'labelAnnual':'int16',
                }
  export_bands = ['B1','B2','B3','B4','B5','B7','temp_K','clouds','label2024','labelAnnual']
  satellite_abrv = 'LS'
elif satellite == 'Sentinel-2':
  im_coll_all = s2_ic
        # .filter(ee.Filter.gte('n_bands', 9))
  # Specify scale
  scale_sel = 10
  cast_dict = {
                'B2':'int16',
                'B3':'int16',
                'B4':'int16',
                'B5':'int16',
                'B8':'int16',
                'B8A':'int16',
                'B11':'int16',
                'B12':'int16',
                'clouds':'int16',
                'label2024':'int16',
                'labelAnnual':'int16',
                }
  export_bands = ['B2','B3','B4','B5','B8','B8A','B11','B12','clouds','label2024','labelAnnual']
  satellite_abrv = 'S2'

# Export for all years
# for i in range(n_features):
for i in range(300,350):
# for i in range(17, 18):
# for i in range(53, 56):
# for i in range(35,36):
  print(f'site {i}')
  # Get mining area name
  name_sel = aoi_names[i]
  # Filter mining polygons to only choose polygon with that name
  aoi_poly_sel = aoi_polys.filter(ee.Filter.eq(site_id_field,name_sel))

  # Defining a point geometry from our lat/long
  # (Data type: Point Geometry)
  aoi = aoi_poly_sel.geometry()
  bounds = aoi.bounds()

  export_buffer = bounds.buffer(500).bounds()

  # Filter to selected area of interest
  im_coll = im_coll_all.filterBounds(aoi)

  # display('first image', im_coll.first())
  # display('Number of images', im_coll.size())

  # Step 2: Map a function to check if the AOI is entirely within the image footprint and set a property.
  def check_containment(image):
    footprint = ee.Geometry(image.geometry())
    contains = footprint.contains(aoi)
    return image.set('contains', contains)

  # Map the function over the collection.
  im_coll = im_coll.map(check_containment)

  # Step 3: Filter the collection based on the 'contains' property.
  im_coll = im_coll.filter(ee.Filter.eq('contains', True))

  # display('Number of images post-containment check', )

  # Label image with clouds and slump labels
  # Slump label from 2024 *and* year after image,
  # since annual labels likely fully contain slump area from previous year
  # (TO DO: Potentially add water or snow as well?)
  def labelImage(image):
    # The sample region is stored as a property
    sample_region = aoi
    clouds = ee.Image(image.select('clouds')).selfMask()
    snow = image.select('B3').gt(4000).rename('snow').selfMask()
    baseline = image.select('B3').gte(0).rename('baseline')
    count = clouds.addBands(baseline).addBands(snow).reduceRegion(
                geometry = sample_region.buffer(100),
                reducer = ee.Reducer.count(),
                scale = 30
                )

    image_year = ee.Number(image.date().get('year')).min(2023).add(1)
    # result = ee.Feature(sample_region.centroid()).set(count)
    result = (image
                  .addBands(aoi_label_mask)
                  .addBands(all_slumps_painted.filter(ee.Filter.eq('year', image_year)).first().rename('labelAnnual'))
                  .set(count)
    )

    return result

  # Get vegetation and cloud counts
  image_export_timeseries = im_coll.map(labelImage).map(clearEnough)
  # display('result', ee.Image(image_export_timeseries.sort('CLOUDY_PIXEL_PERCENTAGE').limit(15)))

  # Only get images with 100% coverage of AOI
  image_export_timeseries = image_export_timeseries.filter(ee.Filter.eq('clear_enough', 1))
  # display('result', ee.Image(image_export_timeseries.limit(4)))

  # Combine images that overlap
  unique_dates = image_export_timeseries.aggregate_array('date').distinct()

  # display(unique_dates)
  def combineDate_imgs(date):
    # same_date_imgs = image_export_timeseries.filter(ee.Filter.eq('date', date))
    # median = ee.Image(same_date_imgs.median()).copyProperties(ee.Image(same_date_imgs.first()))
    # return median
    return image_export_timeseries.filter(ee.Filter.eq('date', date)).first()


  # Test Mapping
  # Map.add_ee_layer(export_buffer, {'color': 'yellow'}, f'export example {i}')
  # Map.add_ee_layer(im_coll.first(), im_vis, f'Landsat img., site {i}')

  # Get export images
  # export_images_fc = ee.ImageCollection(unique_dates.map(combineDate_imgs))


  # export_images_fc = (export_images_fc
  export_images = (image_export_timeseries
                   .sort('date')
                   .select(export_bands)
                   .cast(cast_dict, export_bands)
                  #  .limit(10)
  )

  export_dates = export_images.aggregate_array('date').getInfo()
  # display(export_dates[1:20])
  # export_images = export_images_fc.toList(1000)
  # .toList(10)

  n_export_images = export_images.size().getInfo()
  display(f'num images vs, passing cloud rule: {im_coll.size().getInfo()}/{n_export_images}')
  # display(export_images.getInfo() == [])

  # Export images
  if not export_images.getInfo() == []:
    export_folder = 'taymyr_S2_exports'

    # display(export_images.first())

    # Run function to export images
    export_image_batch(export_images=export_images, export_dates=export_dates,
                       site_name=name_sel, export_folder=export_folder,
                       export_folder_path=f'{wd_imports_imgs}{export_folder}',
                       n_export_images=n_export_images,
                       region=export_buffer,scale=scale_sel,
                       start_tasks=yes_no_image_export)

  print(datetime.now().time())
# Map.centerObject(export_buffer, 11)


In [ ]:
f'{wd_imports_imgs}{export_folder}'

In [ ]:
# print(metadata_file['date'])
print(metadata_file['date'].unique())

(ggplot(metadata_file, aes(x = 'clouds'))
+ geom_histogram())

metadata_file = metadata_file.sort_values(by = ['year','clouds_img'])

metadata_file['best_image'] = (metadata_file
                               .groupby(['year','yday20'])['clouds']
                               .rank(method='first', ascending=True)
)
display(metadata_file.groupby(['year','yday20'])['clouds'].rank(method='first', ascending=True))
metadata_file_best = metadata_file[metadata_file['best_image'] == 1].copy().sort_values(by=['date']).reset_index()
display(metadata_file_best)
# (ggplot(metadata_file, aes(x = 'yday20'))
# +geom_bar(color = 'black', fill = 'black')
# +facet_wrap(['year'], ncol=1)
# )

In [ ]:
export_images_duplicate = export_images_fc.filterDate('2018-08-24','2018-08-25')
display(export_images_duplicate)

In [ ]:
display(export_images_duplicate.aggregate_array('SENSING_ORBIT_NUMBER'))
display(export_images_duplicate.aggregate_array('system:time_start'))
display(export_images_duplicate.aggregate_array('MGRS_TILE'))

In [ ]:
display(len(set(export_dates))/len(export_dates))

In [ ]:
# Export for all years
for i in range(1, len(aoi_names)):
# for i in range(17, 18):
# for i in range(53, 56):
# for i in range(2):
  print(f'site {i}')
  # Get mining area name
  name_sel = aoi_names[i]
  # Filter mining polygons to only choose polygon with that name
  aoi_poly_sel = aoi_polys.filter(ee.Filter.eq(site_id_field,name_sel))

  # Defining a point geometry from our lat/long
  # (Data type: Point Geometry)
  aoi = aoi_poly_sel.geometry()
  bounds = aoi.bounds()

  export_buffer = bounds.buffer(500).bounds()

  # Filter to selected area of interest
  im_coll = im_coll_all.filterBounds(aoi)

  # display('first image', im_coll.first())
  # display('Number of images', im_coll.size())

  # Step 2: Map a function to check if the AOI is entirely within the image footprint and set a property.
  def check_containment(image):
    footprint = ee.Geometry(image.geometry())
    contains = footprint.contains(aoi)
    return image.set('contains', contains)

  # Map the function over the collection.
  im_coll = im_coll.map(check_containment)

  # Step 3: Filter the collection based on the 'contains' property.
  im_coll = im_coll.filter(ee.Filter.eq('contains', True))

  # display('Number of images after second filter', im_coll.size())

  # Add NDVI
  veg_coll = im_coll.map(getClouds)
  # display('NDVI collection', veg_coll)

  # Specify scale
  scale_sel = 90

  # Get number of images at polygon
  def getCount(image):
    # The sample region is stored as a property
    sample_region = aoi

    count = image.reduceRegion(
                geometry = sample_region,
                reducer = ee.Reducer.count(),
                scale = scale_sel
                )

    # result = ee.Feature(sample_region.centroid()).set(count)
    result = (ee.Feature(None)
                  .set(count)
                  .set('site_no', name_sel, 'id', image.id())
                  .copyProperties(image, ['date','month','year', 'WRS_PATH'])
    )

    return result

  image_export_timeseries = veg_coll.map(getCount).map(clearEnough)

  # Only get images with 100% coverage of mining area
  image_export_timeseries = image_export_timeseries.filter(ee.Filter.eq('clear_enough', 1))
  display('result', image_export_timeseries.first())

  # Make a list of image IDs
  clear_img_list = image_export_timeseries.aggregate_array('id')
  # display(clear_img_list)

  # Export the timeseries table
  table_export = ee.batch.Export.table.toDrive(
      collection=image_export_timeseries,
      description=f'mining_timeseries_{name_sel}_test',
      fileNamePrefix=f'mining_timeseries_{name_sel}_test',
      fileFormat='CSV',
      folder='mining_ndvi_timeseries'
  )

  # Get export images
  export_images = (im_coll
                   .filter(ee.Filter.inList('system:index', clear_img_list))
                  #  .filter(ee.Filter.lt('CLOUD_COVER', 30))
                  #  .sort('CLOUD_COVER')
                   .sort('year')
                   .select(['B1','B2','B3','B4','B5','B6','B7'])
                  #  .limit(10)
  ).toList(1000)
  # .toList(10)


  print(len(export_images.getInfo()))
  display(export_images.getInfo() == [])

  # Export images
  if not export_images.getInfo() == []:
    if num_export_imgs == 'pair':
      export_images = ee.List([
        ee.Image(export_images.get(0)),
        ee.Image(export_images.get(-1))
      ])
      export_folder = 'mining_ndvi_before_after'
    else:
      export_folder = 'mining_ndvi_timeseries_images'

    display(export_images.get(0))
    # display(export_images)
    # Loop over the list and export each image.
    for i in range(export_images.size().getInfo()):
      export_image = ee.Image(export_images.get(i))

      # image_year = export_image.get('year').getInfo()
      image_date = export_image.date().format('YYYYMMdd').getInfo()
      # Define the export task.
      if img_type == 'vis':
        export_image = export_image.visualize(**im_vis)
      image_export = ee.batch.Export.image.toDrive(
          image=export_image,
          description=f'Landsat_Image_{name_sel}_{image_date}',
          fileNamePrefix=f'Landsat_Image_{name_sel}_{image_date}',
          folder=export_folder,
          scale=30,
          region=export_buffer,
          crs='EPSG:3857',
          fileFormat='GeoTIFF'
      )
      if yes_no_image_export:
        # Start the image export task.
        image_export.start()
        # print(f'Exporting Landsat Image {name_sel} to Google Drive.')

  if yes_no_table_export:
    # Start the table export task
    table_export.start()
    print(f'Exporting table {name_sel} to Google Drive.')